In [2]:
import pandas as pd

CLUSTER_LABELS_CSV = "clustering/twitter_hdbscan_cluster_labels.csv"

cluster_labels_df = pd.read_csv(CLUSTER_LABELS_CSV, usecols=['cluster', 'labels'])
cluster_labels_map = {-1: 'Noise'}
cluster_labels_map.update(dict(zip(cluster_labels_df['cluster'], cluster_labels_df['labels'])))
print(cluster_labels_map)

{-1: 'Noise', 0: "Mother's Day greetings", 1: 'Birthdays and birthday wishes', 2: 'Boredom / nothing to do', 3: 'Books and reading', 4: 'Sports talk', 5: 'Drinking / alcohol', 6: 'Food, coffee, and eating', 7: 'School and exams', 8: 'Missing someone / emotional longing', 9: 'Haircuts and hair', 10: 'Fashion and clothes', 11: 'Pets / dogs and cats', 12: 'Weather', 13: 'Photos and pictures', 14: 'Star Wars Day', 15: 'Feeling sick / health complaints', 16: 'Phones and tech issues', 17: 'Followers / follow-back Twitter activity', 18: 'Twitter and tweeting', 19: 'Gratitude / thanks / appreciation', 20: 'Sad feelings / negative mood', 21: 'Good morning greetings', 22: 'Staying home / nighttime routine', 23: 'Jobs and work search', 24: 'Money and bills', 25: 'Music and songs', 26: 'Movies / Star Trek', 27: 'Sad or emotional chatter', 28: 'Sleep and tiredness', 29: 'Work and weekend anticipation', 30: 'Cars, driving, and traffic', 31: 'Concert tickets and live events', 32: 'Wanting to go out /

In [14]:
from datasets import load_from_disk

clustered_csv = "clustering/twitter_hdbscan_clustered.csv"
clustered_df = pd.read_csv(clustered_csv)
print(f"Size of clustered CSV: {len(clustered_df)}")

predicted_test_csv = "results/roberta_base_test_predictions.csv"
predicted_test_df = pd.read_csv(predicted_test_csv)
print(f"Size of test dataset: {len(predicted_test_df)}")

# If row is in predicted_test_df, keep in clustered_df, else drop
print('-' * 80)
test_clustered_df = clustered_df[clustered_df["textID"].isin(predicted_test_df["textID"])]
print(f"Size of test_clustered_df: {len(test_clustered_df)}")
print(test_clustered_df.head())

Size of clustered CSV: 30889
Size of test dataset: 3534
--------------------------------------------------------------------------------
Size of test_clustered_df: 3504
           textID                                               text  \
27385  f87dea47db  Last session of the day  http://twitpic.com/67ezh   
27386  96d74cb729   Shanghai is also really exciting (precisely -...   
27387  eee518ae67  Recession hit Veronique Branquinho, she has to...   
27388  01082688c6                                        happy bday!   
27389  33987a8ee5             http://twitpic.com/4w75p - I like it!!   

      sentiment                                           raw_text  \
27385   neutral  Last session of the day  http://twitpic.com/67ezh   
27386  positive   Shanghai is also really exciting (precisely -...   
27387  negative  Recession hit Veronique Branquinho, she has to...   
27388  positive                                        happy bday!   
27389  positive             http://twitpic.com/4

In [12]:
# Find the 30 missing rows
missing_df = predicted_test_df[~predicted_test_df["textID"].isin(test_clustered_df["textID"])]
print(f"Size of missing_df: {len(missing_df)} -- Expect 30")
print(missing_df.head(10))

Size of missing_df: 30 -- Expect 30
         textID  predicted_label_id predicted_sentiment  \
16   9fce30159a                   0            negative   
91   3a98aa4762                   2            positive   
146  f254748cdb                   2            positive   
194  f7835156a5                   2            positive   
253  2c72a5582a                   1             neutral   
333  3bd0bff670                   2            positive   
417  226adfd977                   2            positive   
661  bcf13877f7                   2            positive   
890  4477b985e5                   1             neutral   
994  fe09df7e91                   0            negative   

                                                 text  
16                                           Miss you  
91                               Happy Mothers Day!!!  
146                                 Happy Mothers Day  
194                                             woot!  
253  WHAT ABOUT ME ??  I VOTE EVER

The above 30 rows were probably dropped due to duplicate content after post-processing in clustering.

In [15]:
merged_df = test_clustered_df.merge(
    predicted_test_df[["textID", "predicted_sentiment"]],
    on="textID",
    how="left"
)
print(f"Size of merged_df: {len(merged_df)}")
print(merged_df.head())

Size of merged_df: 3504
       textID                                               text sentiment  \
0  f87dea47db  Last session of the day  http://twitpic.com/67ezh   neutral   
1  96d74cb729   Shanghai is also really exciting (precisely -...  positive   
2  eee518ae67  Recession hit Veronique Branquinho, she has to...  negative   
3  01082688c6                                        happy bday!  positive   
4  33987a8ee5             http://twitpic.com/4w75p - I like it!!  positive   

                                            raw_text  \
0  Last session of the day  http://twitpic.com/67ezh   
1   Shanghai is also really exciting (precisely -...   
2  Recession hit Veronique Branquinho, she has to...   
3                                        happy bday!   
4             http://twitpic.com/4w75p - I like it!!   

                                          clean_text  cluster  \
0                            last session of the day       -1   
1  shanghai is also really exciting (pre

In [25]:
from sklearn.metrics import f1_score, cohen_kappa_score

required_cols = ["cluster", "sentiment", "predicted_sentiment"]
eval_df = merged_df[required_cols].dropna().copy()


def per_cluster_metrics(group):
    y_true = group[true_col]
    y_pred = group["predicted_sentiment"]
    return pd.Series(
        {
            "n_samples": len(group),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        }
    )


cluster_f1_df = eval_df.groupby("cluster").apply(per_cluster_metrics).reset_index()
cluster_f1_df["cluster_label"] = cluster_f1_df["cluster"].map(cluster_labels_map).fillna("Unknown")
cluster_f1_df = cluster_f1_df[["cluster", "cluster_label", "n_samples", "f1_macro", "f1_weighted", "qwk"]]
cluster_f1_df = cluster_f1_df.sort_values(["f1_macro", "n_samples"], ascending=[False, False]).reset_index(drop=True)

print(f"Using ground-truth column: {true_col}")
print(f"Total rows evaluated: {len(eval_df)}")
print(cluster_f1_df.round(4).to_string(index=False))

cluster_f1_df

Using ground-truth column: sentiment
Total rows evaluated: 3504
 cluster                            cluster_label  n_samples  f1_macro  f1_weighted    qwk
       1            Birthdays and birthday wishes       27.0    0.9339       0.9259 0.9049
      32    Wanting to go out / wishing for plans       27.0    0.9280       0.9259 0.9375
      13                      Photos and pictures       28.0    0.8518       0.8568 0.7710
      28                      Sleep and tiredness       71.0    0.8467       0.8432 0.8724
      24                          Money and bills       13.0    0.8452       0.8462 0.6905
      26                       Movies / Star Trek       75.0    0.8420       0.8428 0.8552
       7                         School and exams       95.0    0.8247       0.8205 0.8067
      11                     Pets / dogs and cats       55.0    0.8133       0.7999 0.7552
      29            Work and weekend anticipation       33.0    0.8038       0.8210 0.7485
      18                  

,cluster,cluster_label,n_samples,f1_macro,f1_weighted,qwk
0,1,Birthdays and birthday wishes,27.0,0.933862,0.925926,0.904930
1,32,Wanting to go out / wishing for plans,27.0,0.928030,0.925926,0.937500
2,13,Photos and pictures,28.0,0.851813,0.856779,0.771028
3,28,Sleep and tiredness,71.0,0.846688,0.843195,0.872406
4,24,Money and bills,13.0,0.845238,0.846154,0.690476
5,26,Movies / Star Trek,75.0,0.841981,0.842844,0.855212
6,7,School and exams,95.0,0.824653,0.820494,0.806723
7,11,Pets / dogs and cats,55.0,0.813272,0.799916,0.755245
8,29,Work and weekend anticipation,33.0,0.803812,0.820979,0.748518
9,18,Twitter and tweeting,116.0,0.802071,0.801844,0.752665


## 1. What `f1_weighted` means in sklearn

In multiclass classification, sklearn computes F1 per class, then aggregates.

- **Per-class F1:**
  
  F1_i = 2 * (precision_i * recall_i) / (precision_i + recall_i)

- **Weighted F1:**
  
  F1_weighted = Σ (n_i / N) * F1_i

Where:
- n_i = number of true samples in class i (support)
- N = total number of samples

**Key idea:**  
Each class contributes proportionally based on how frequent it is.

---

## 2. Contrast with `f1_macro`

- **Macro F1:**
  
  F1_macro = (1 / C) * Σ F1_i

- Treats all classes equally
- Ignores class imbalance

---

## 3. Practical interpretation

| Metric         | What it emphasizes                  |
|----------------|------------------------------------|
| Macro F1       | Equal performance across classes   |
| Weighted F1    | Performance on majority classes    |

---

## 4. Impact on results

Example clusters:

| Cluster        | Macro F1 | Weighted F1 |
|----------------|----------|-------------|
| Gratitude      | 0.51     | 0.90        |
| Good morning   | 0.59     | 0.88        |
| Mother’s Day   | 0.62     | 0.86        |

### Interpretation

- Model predicts the dominant sentiment very well (likely positive for these 3 examples).
- Performs poorly on minority sentiments.
- The model is **biased** toward majority sentiment within each topic.

---



In [27]:
# Topics where weighted F1 and macro F1 differ by more than 0.1
f1_gap_df = cluster_f1_df.copy()
f1_gap_df["f1_gap"] = f1_gap_df["f1_weighted"] - f1_gap_df["f1_macro"]
f1_gap_df["abs_f1_gap"] = f1_gap_df["f1_gap"].abs()

topics_large_gap_df = (
    f1_gap_df[f1_gap_df["abs_f1_gap"] > 0.1]
    .sort_values("abs_f1_gap", ascending=False)
    .reset_index(drop=True)
)

print(f"Topics with |weighted F1 - macro F1| > 0.1: {len(topics_large_gap_df)}")
print(
    topics_large_gap_df[
        ["cluster", "cluster_label", "n_samples", "f1_macro", "f1_weighted", "f1_gap", "abs_f1_gap", "qwk"]
    ].round(4).to_string(index=False)
)

topics_large_gap_df

Topics with |weighted F1 - macro F1| > 0.1: 4
 cluster                     cluster_label  n_samples  f1_macro  f1_weighted  f1_gap  abs_f1_gap    qwk
      19 Gratitude / thanks / appreciation       47.0    0.5083       0.9007  0.3924      0.3924 0.3160
      21            Good morning greetings       21.0    0.5892       0.8817  0.2925      0.2925 0.8125
       0            Mother's Day greetings      106.0    0.6203       0.8639  0.2436      0.2436 0.6906
      15  Feeling sick / health complaints      123.0    0.7111       0.8164  0.1053      0.1053 0.6680


,cluster,cluster_label,n_samples,f1_macro,f1_weighted,qwk,f1_gap,abs_f1_gap
0,19,Gratitude / thanks / appreciation,47.0,0.508306,0.900686,0.316008,0.392380,0.392380
1,21,Good morning greetings,21.0,0.589247,0.881720,0.812500,0.292473,0.292473
2,0,Mother's Day greetings,106.0,0.620348,0.863939,0.690624,0.243591,0.243591
3,15,Feeling sick / health complaints,123.0,0.711088,0.816365,0.667986,0.105278,0.105278


In [20]:
# Confusion matrix for each cluster with |weighted F1 - macro F1| > 0.1
from sklearn.metrics import confusion_matrix

# Recompute if needed, otherwise reuse earlier result.
if "topics_large_gap_df" not in globals() or topics_large_gap_df.empty:
    f1_gap_df = cluster_f1_df.copy()
    f1_gap_df["f1_gap"] = f1_gap_df["f1_weighted"] - f1_gap_df["f1_macro"]
    f1_gap_df["abs_f1_gap"] = f1_gap_df["f1_gap"].abs()
    topics_large_gap_df = (
        f1_gap_df[f1_gap_df["abs_f1_gap"] > 0.1]
        .sort_values("abs_f1_gap", ascending=False)
        .reset_index(drop=True)
    )

selected_clusters = topics_large_gap_df["cluster"].tolist()

if len(selected_clusters) == 0:
    print("No clusters found with |weighted F1 - macro F1| > 0.1")
else:
    print(f"Building confusion matrices for {len(selected_clusters)} high-gap clusters...\n")

    for cluster_id in selected_clusters:
        cluster_name = cluster_labels_map.get(cluster_id, "Unknown")
        subset = eval_df[eval_df["cluster"] == cluster_id]

        y_true = subset[true_col]
        y_pred = subset["predicted_sentiment"]

        labels = sorted(set(y_true.unique()) | set(y_pred.unique()))
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        cm_df = pd.DataFrame(cm, index=labels, columns=labels)

        print(f"Cluster {cluster_id} - {cluster_name} (n={len(subset)})")
        print("Rows = true sentiment, Columns = predicted sentiment")
        print(cm_df.to_string())
        print("-" * 80)

# Optional: keep these clusters as a quick reference table
topics_large_gap_df[["cluster", "cluster_label", "n_samples", "f1_macro", "f1_weighted", "abs_f1_gap"]]

Building confusion matrices for 4 high-gap clusters...

Cluster 19 - Gratitude / thanks / appreciation (n=47)
Rows = true sentiment, Columns = predicted sentiment
          negative  neutral  positive
negative         0        0         1
neutral          0        2         2
positive         0        1        41
--------------------------------------------------------------------------------
Cluster 21 - Good morning greetings (n=21)
Rows = true sentiment, Columns = predicted sentiment
          negative  neutral  positive
negative         0        1         0
neutral          0        4         1
positive         0        0        15
--------------------------------------------------------------------------------
Cluster 0 - Mother's Day greetings (n=106)
Rows = true sentiment, Columns = predicted sentiment
          negative  neutral  positive
negative         1        1         0
neutral          3       12         2
positive         0       10        77
---------------------------

,cluster,cluster_label,n_samples,f1_macro,f1_weighted,abs_f1_gap
0,19,Gratitude / thanks / appreciation,47.0,0.508306,0.900686,0.392380
1,21,Good morning greetings,21.0,0.589247,0.881720,0.292473
2,0,Mother's Day greetings,106.0,0.620348,0.863939,0.243591
3,15,Feeling sick / health complaints,123.0,0.711088,0.816365,0.105278


In [23]:
# Additional metrics (including Quadratic Weighted Kappa / QWK) for each high-gap cluster
from sklearn.metrics import accuracy_score, precision_score, recall_score, cohen_kappa_score

# Use the same clusters identified by the F1-gap analysis.
if "topics_large_gap_df" not in globals() or topics_large_gap_df.empty:
    raise ValueError("Run Cell 8 first to compute topics_large_gap_df.")

qwk_rows = []
for cluster_id in topics_large_gap_df["cluster"].tolist():
    subset = eval_df[eval_df["cluster"] == cluster_id]
    y_true = subset[true_col]
    y_pred = subset["predicted_sentiment"]

    qwk_rows.append(
        {
            "cluster": cluster_id,
            "cluster_label": cluster_labels_map.get(cluster_id, "Unknown"),
            "n_samples": len(subset),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "qwk": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        }
    )

qwk_metrics_df = pd.DataFrame(qwk_rows).sort_values("qwk", ascending=False).reset_index(drop=True)

print("Per-cluster QWK and supporting metrics (high F1-gap clusters):")
print(qwk_metrics_df.round(4).to_string(index=False))

# Overall metrics across just these four clusters
high_gap_eval_df = eval_df[eval_df["cluster"].isin(topics_large_gap_df["cluster"])].copy()
overall_qwk = cohen_kappa_score(high_gap_eval_df[true_col], high_gap_eval_df["predicted_sentiment"], weights="quadratic")
overall_accuracy = accuracy_score(high_gap_eval_df[true_col], high_gap_eval_df["predicted_sentiment"])

print("\nOverall (combined four high-gap clusters):")
print(f"Accuracy: {overall_accuracy:.4f}")
print(f"QWK: {overall_qwk:.4f}")

qwk_metrics_df

Per-cluster QWK and supporting metrics (high F1-gap clusters):
 cluster                     cluster_label  n_samples  accuracy  precision_macro  recall_macro  precision_weighted  recall_weighted    qwk
      21            Good morning greetings         21    0.9048           0.5792        0.6000              0.8601           0.9048 0.8125
       0            Mother's Day greetings        106    0.8491           0.5821        0.6970              0.8884           0.8491 0.6906
      15  Feeling sick / health complaints        123    0.8293           0.8044        0.6634              0.8224           0.8293 0.6680
      19 Gratitude / thanks / appreciation         47    0.9149           0.5328        0.4921              0.8894           0.9149 0.3160

Overall (combined four high-gap clusters):
Accuracy: 0.8552
QWK: 0.8820


,cluster,cluster_label,n_samples,accuracy,precision_macro,recall_macro,precision_weighted,recall_weighted,qwk
0,21,Good morning greetings,21,0.904762,0.579167,0.600000,0.860119,0.904762,0.812500
1,0,Mother's Day greetings,106,0.849057,0.582141,0.696980,0.888368,0.849057,0.690624
2,15,Feeling sick / health complaints,123,0.829268,0.804413,0.663420,0.822369,0.829268,0.667986
3,19,Gratitude / thanks / appreciation,47,0.914894,0.532828,0.492063,0.889426,0.914894,0.316008


In [ ]:
# Extract clusters with sufficient support but low QWK
MIN_SAMPLES = 30
LOW_QWK_THRESHOLD = 0.70

if "qwk_all_clusters_df" not in globals():
    from sklearn.metrics import cohen_kappa_score

    qwk_all_rows = []
    for cluster_id in sorted(eval_df["cluster"].unique()):
        subset = eval_df[eval_df["cluster"] == cluster_id]
        qwk_all_rows.append(
            {
                "cluster": cluster_id,
                "cluster_label": cluster_labels_map.get(cluster_id, "Unknown"),
                "n_samples": len(subset),
                "qwk": cohen_kappa_score(subset[true_col], subset["predicted_sentiment"], weights="quadratic"),
            }
        )
    qwk_all_clusters_df = pd.DataFrame(qwk_all_rows)

low_qwk_sufficient_df = (
    qwk_all_clusters_df[
        (qwk_all_clusters_df["n_samples"] >= MIN_SAMPLES)
        & (qwk_all_clusters_df["qwk"] < LOW_QWK_THRESHOLD)
    ]
    .sort_values(["qwk", "n_samples"], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"Thresholds -> MIN_SAMPLES: {MIN_SAMPLES}, LOW_QWK_THRESHOLD: {LOW_QWK_THRESHOLD}")
print(f"Clusters matched: {len(low_qwk_sufficient_df)}")
print(low_qwk_sufficient_df.round(4).to_string(index=False))

low_qwk_sufficient_df

Thresholds -> MIN_SAMPLES: 30, LOW_QWK_THRESHOLD: 0.7
Clusters matched: 5
 cluster                       cluster_label  n_samples    qwk                             plot_label
      19   Gratitude / thanks / appreciation         47 0.3160  19: Gratitude / thanks / appreciation
       8 Missing someone / emotional longing         30 0.5902 8: Missing someone / emotional longing
      15    Feeling sick / health complaints        123 0.6680   15: Feeling sick / health complaints
      16              Phones and tech issues        118 0.6697             16: Phones and tech issues
       0              Mother's Day greetings        106 0.6906              0: Mother's Day greetings


,cluster,cluster_label,n_samples,qwk,plot_label
0,19,Gratitude / thanks / appreciation,47,0.316008,19: Gratitude / thanks / appreciation
1,8,Missing someone / emotional longing,30,0.590164,8: Missing someone / emotional longing
2,15,Feeling sick / health complaints,123,0.667986,15: Feeling sick / health complaints
3,16,Phones and tech issues,118,0.669704,16: Phones and tech issues
4,0,Mother's Day greetings,106,0.690624,0: Mother's Day greetings
